# PCY Algorithm — Park-Chen-Yu

## Learning Objectives

1. **Explain** why A-Priori's Pass 2 can run out of memory for candidate pairs
2. **Describe** how PCY uses a hash table during Pass 1 to eliminate infrequent pair buckets
3. **Define** the bitmap trick and how it frees memory for Pass 2 candidate counts
4. **Prove** the PCY guarantee: infrequent bucket ⟹ pair infrequent
5. **Implement** PCY for frequent pair mining


## Problem Statement

### A-Priori's Memory Bottleneck

A-Priori's Pass 2 must count *every* candidate pair. If $m$ items pass the singleton threshold, there are $\binom{m}{2} = O(m^2)$ candidate pairs. For $m = 10^5$ frequent items, this is $5 \times 10^9$ pairs — about 20 GB at 4 bytes each. This will not fit in RAM.

### PCY's Key Insight

During Pass 1 (when we count singletons anyway), we have spare memory in RAM. Use this spare memory for a **hash table** to count how many pairs hash to each bucket. After Pass 1, any bucket whose count is below the support threshold $s$ *cannot contain any frequent pair* (by the subset monotonicity of support). We convert the hash table to a compact **bitmap** (1 bit per bucket) and use it in Pass 2 to skip candidate pairs in infrequent buckets.


In [ ]:
from IPython.display import SVG, display

svg = '''
<svg xmlns="http://www.w3.org/2000/svg" width="820" height="360" font-family="monospace" font-size="12">
  <rect width="820" height="360" fill="#fafafa" rx="8"/>
  <defs><marker id="arr" markerWidth="8" markerHeight="6" refX="7" refY="3" orient="auto"><polygon points="0 0,8 3,0 6" fill="#999"/></marker></defs>

  <!-- PASS 1 -->
  <text x="140" y="22" text-anchor="middle" fill="#333" font-size="12" font-weight="bold">Pass 1: Count items + Hash pairs</text>
  <rect x="10" y="35" width="260" height="160" rx="4" fill="#e8f0fb" stroke="#4e79a7" stroke-width="1.5"/>

  <text x="140" y="56" text-anchor="middle" fill="#4e79a7" font-size="11">For each basket B:</text>
  <text x="140" y="74" text-anchor="middle" fill="#333" font-size="11">① Count each item  →  C₁[item]++</text>
  <text x="140" y="92" text-anchor="middle" fill="#333" font-size="11">② For each pair {i,j} ⊂ B:</text>
  <text x="140" y="110" text-anchor="middle" fill="#333" font-size="11">   bucket = h(i,j) mod B</text>
  <text x="140" y="128" text-anchor="middle" fill="#333" font-size="11">   hash_table[bucket]++</text>
  <text x="140" y="155" text-anchor="middle" fill="#555" font-size="10">Memory split between item counts</text>
  <text x="140" y="170" text-anchor="middle" fill="#555" font-size="10">and hash table (both in RAM)</text>

  <!-- Hash table SVG -->
  <rect x="10" y="205" width="260" height="100" rx="4" fill="white" stroke="#ccc" stroke-width="1"/>
  <text x="140" y="222" text-anchor="middle" fill="#555" font-size="11" font-weight="bold">Hash Table (counts per bucket)</text>
  <rect x="25"  y="230" width="30" height="22" fill="#59a14f" rx="2"/><text x="40"  y="246" text-anchor="middle" fill="white" font-size="10">12</text>
  <rect x="60"  y="230" width="30" height="22" fill="#59a14f" rx="2"/><text x="75"  y="246" text-anchor="middle" fill="white" font-size="10">8</text>
  <rect x="95"  y="230" width="30" height="22" fill="#e15759" rx="2"/><text x="110" y="246" text-anchor="middle" fill="white" font-size="10">3</text>
  <rect x="130" y="230" width="30" height="22" fill="#59a14f" rx="2"/><text x="145" y="246" text-anchor="middle" fill="white" font-size="10">15</text>
  <rect x="165" y="230" width="30" height="22" fill="#e15759" rx="2"/><text x="180" y="246" text-anchor="middle" fill="white" font-size="10">2</text>
  <rect x="200" y="230" width="30" height="22" fill="#59a14f" rx="2"/><text x="215" y="246" text-anchor="middle" fill="white" font-size="10">9</text>
  <rect x="235" y="230" width="30" height="22" fill="#e15759" rx="2"/><text x="250" y="246" text-anchor="middle" fill="white" font-size="10">1</text>
  <line x1="10" y1="260" x2="270" y2="260" stroke="#ddd" stroke-width="0.5"/>
  <text x="140" y="278" text-anchor="middle" fill="#555" font-size="10">green ≥ s → frequent bucket</text>
  <text x="140" y="293" text-anchor="middle" fill="#555" font-size="10">red &lt; s → prune all pairs in bucket</text>

  <!-- BITMAP -->
  <text x="430" y="22" text-anchor="middle" fill="#333" font-size="12" font-weight="bold">Bitmap + Pass 2</text>
  <rect x="290" y="35" width="240" height="100" rx="4" fill="#fff4e0" stroke="#f28e2b" stroke-width="1.5"/>
  <text x="410" y="55" text-anchor="middle" fill="#f28e2b" font-size="11" font-weight="bold">Bitmap (1 bit per bucket)</text>
  <text x="410" y="75" text-anchor="middle" fill="#333" font-size="11">bucket_count ≥ s  →  bit = 1</text>
  <text x="410" y="93" text-anchor="middle" fill="#333" font-size="11">bucket_count &lt; s  →  bit = 0</text>
  <text x="410" y="115" text-anchor="middle" fill="#555" font-size="10">Replaces full hash table: 1 bit vs 4 bytes</text>
  <text x="410" y="130" text-anchor="middle" fill="#555" font-size="10">saves memory for Pass 2 candidate counts</text>

  <rect x="290" y="145" width="240" height="115" rx="4" fill="#e8f8e8" stroke="#59a14f" stroke-width="1.5"/>
  <text x="410" y="163" text-anchor="middle" fill="#59a14f" font-size="11" font-weight="bold">Pass 2: Count candidates C₂</text>
  <text x="410" y="181" text-anchor="middle" fill="#333" font-size="11">Pair {i,j} is a candidate iff:</text>
  <text x="410" y="199" text-anchor="middle" fill="#333" font-size="12" font-weight="bold">① C₁[i] ≥ s  AND  C₁[j] ≥ s</text>
  <text x="410" y="217" text-anchor="middle" fill="#333" font-size="12" font-weight="bold">② bitmap[ h(i,j) ] = 1</text>
  <text x="410" y="243" text-anchor="middle" fill="#555" font-size="10">Condition ② eliminates pairs whose bucket</text>
  <text x="410" y="258" text-anchor="middle" fill="#555" font-size="10">was infrequent → cannot be frequent</text>

  <!-- Key insight -->
  <rect x="545" y="35" width="265" height="225" rx="6" fill="#f5f5f5" stroke="#ccc" stroke-width="1.5"/>
  <text x="677" y="55" text-anchor="middle" fill="#333" font-size="11" font-weight="bold">Why PCY Saves Memory</text>
  <line x1="545" y1="62" x2="810" y2="62" stroke="#ccc" stroke-width="0.5"/>
  <text x="677" y="80" text-anchor="middle" fill="#555" font-size="11">A-Priori Pass 2 must count</text>
  <text x="677" y="96" text-anchor="middle" fill="#555" font-size="11">ALL O(m²) candidate pairs.</text>
  <text x="677" y="118" text-anchor="middle" fill="#555" font-size="11">PCY Pass 1 hashes pairs to</text>
  <text x="677" y="134" text-anchor="middle" fill="#555" font-size="11">buckets &amp; discards infrequent</text>
  <text x="677" y="150" text-anchor="middle" fill="#555" font-size="11">bucket pairs before Pass 2.</text>
  <text x="677" y="172" text-anchor="middle" fill="#333" font-size="11" font-weight="bold">Memory freed by bitmap:</text>
  <text x="677" y="190" text-anchor="middle" fill="#555" font-size="11">B buckets × 4 bytes → B bits</text>
  <text x="677" y="208" text-anchor="middle" fill="#555" font-size="11">→ 32× smaller</text>
  <text x="677" y="228" text-anchor="middle" fill="#555" font-size="11">→ more RAM for C₂ counts</text>

  <!-- theorem box -->
  <rect x="10" y="310" width="800" height="42" rx="5" fill="#e8f8e8" stroke="#59a14f" stroke-width="1.5"/>
  <text x="410" y="328" text-anchor="middle" fill="#333" font-size="11" font-weight="bold">PCY Guarantee: if bucket_count[h(i,j)] &lt; s  then  count({i,j}) &lt; s  (infrequent)</text>
  <text x="410" y="345" text-anchor="middle" fill="#555" font-size="10">Because: count({i,j}) ≤ bucket_count[h(i,j)] — the pair is a subset of all pairs hashing to same bucket</text>
</svg>
'''

display(SVG(svg))


## Derivation

### Pass 1 Memory Layout

RAM is partitioned between:
- **Item count table:** $O(m)$ integers — one per distinct item
- **PCY hash table:** $B$ integer buckets (rest of available RAM)

Larger $B$ → fewer hash collisions → more pairs eliminated.

### PCY Guarantee

**Lemma:** If $\text{bucket\_count}[h(i,j)] < s$, then $\text{count}(\{i,j\}) < s$.

**Proof:** The support of pair $\{i,j\}$ cannot exceed the count of the bucket it hashes to, because every basket contributing to $\{i,j\}$'s count also contributes to bucket $h(i,j)$'s count. $\square$

### Bitmap Compression

Replace the hash table (4 bytes per bucket) with a bitmap (1 bit per bucket):

$$\text{bitmap}[b] = \begin{cases} 1 & \text{if bucket\_count}[b] \geq s \\ 0 & \text{otherwise} \end{cases}$$

A 32× compression factor frees RAM for storing more candidate pair counts in Pass 2.

### Pass 2 Candidate Filter

Pair $\{i, j\}$ is counted in Pass 2 only if:
1. Both $i$ and $j$ are frequent (their counts passed threshold in Pass 1)
2. $\text{bitmap}[h(i,j)] = 1$ (their bucket was frequent)

Both conditions are necessary; condition 2 is PCY's additional filter over A-Priori.

### Multistage Extension

Run additional passes with independent hash functions before the counting pass:

- Pass $k$: build bitmap$_k$ from pairs surviving all previous bitmaps
- A pair is a candidate only if it passes **all** bitmaps

Each additional pass reduces the candidate set further at the cost of one more read of the data.


## Algorithm Steps

### Inputs
- Baskets, support threshold $s$, number of buckets $B$

### Steps

1. **Pass 1**
   - For each basket $B_t$: increment `item_count[i]` for each item; hash each pair $(i,j)$ → `hash_table[h(i,j)]++`
   - Identify frequent items $L_1 = \{i : \text{count}[i] \geq s\}$
   - Convert hash table to bitmap

2. **Pass 2**
   - For each basket $B_t$, consider only items in $L_1$
   - For each pair $(i, j)$ from those items: if `bitmap[h(i,j)] == 1` → increment `pair_count[{i,j}]`

3. **Output** frequent pairs $L_2 = \{\{i,j\} : \text{pair\_count}[\{i,j\}] \geq s\}$


In [ ]:
from collections import defaultdict
from itertools import combinations


def pcy(baskets, min_support, n_buckets=1000):
    """
    PCY (Park-Chen-Yu) algorithm for frequent pair mining.

    Inputs
    ------
    baskets     : list of sets
    min_support : int
    n_buckets   : int — size of hash table (number of buckets)

    Output
    ------
    frequent_pairs : dict {frozenset: count}
    """
    # ── Pass 1: count items and hash pairs ────────────────────────────────────
    item_count  = defaultdict(int)
    hash_table  = defaultdict(int)

    for basket in baskets:
        for item in basket:
            item_count[item] += 1
        for i, j in combinations(sorted(basket), 2):
            bucket = hash(frozenset([i, j])) % n_buckets
            hash_table[bucket] += 1

    # Frequent items after pass 1
    frequent_items = {item for item, cnt in item_count.items() if cnt >= min_support}

    # Bitmap: 1 iff bucket count >= min_support
    bitmap = {b: (cnt >= min_support) for b, cnt in hash_table.items()}

    # ── Pass 2: count candidate pairs ─────────────────────────────────────────
    pair_count = defaultdict(int)

    for basket in baskets:
        freq_in_basket = [item for item in basket if item in frequent_items]
        for i, j in combinations(sorted(freq_in_basket), 2):
            bucket = hash(frozenset([i, j])) % n_buckets
            # PCY filter: only count if bitmap bit is set
            if bitmap.get(bucket, False):
                pair_count[frozenset([i, j])] += 1

    frequent_pairs = {pair: cnt for pair, cnt in pair_count.items() if cnt >= min_support}
    return frequent_pairs


# ── Demo ──────────────────────────────────────────────────────────────────────
import random
random.seed(42)

items = list("ABCDEFGHIJ")
baskets = [set(random.sample(items, random.randint(3, 6))) for _ in range(500)]

frequent_pairs = pcy(baskets, min_support=50, n_buckets=200)

print(f"Frequent pairs (support ≥ 50):")
for pair, count in sorted(frequent_pairs.items(), key=lambda x: -x[1])[:10]:
    print(f"  {set(pair)}  ->  support = {count}")
